In [1]:
# W12D5: Install dependencies and create project structure

%pip install -q fastapi uvicorn scikit-learn pytest httpx ruff

from pathlib import Path

project_dir = Path("/content/W12D5_Production_AI_System")
project_dir.mkdir(parents=True, exist_ok=True)

(project_dir / "tests").mkdir(exist_ok=True)
(project_dir / "outputs").mkdir(exist_ok=True)
(project_dir / ".github" / "workflows").mkdir(
    parents=True, exist_ok=True
)

print("W12D5 project structure created successfully!")
print("Project directory:", project_dir)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 60.5 MB/s eta 0:00:00
W12D5 project structure created successfully!
Project directory: /content/W12D5_Production_AI_System


In [2]:
# W12D5: Production ML API

app_code = '''
import time
from fastapi import FastAPI, Request
from pydantic import BaseModel
from sklearn.linear_model import LogisticRegression

app = FastAPI(title="W12D5 Production AI System")

# Train a simple demonstration model
X = [[0], [1], [2], [3], [4], [5]]
y = [0, 0, 0, 1, 1, 1]

model = LogisticRegression()
model.fit(X, y)

# Basic in-memory monitoring metrics
metrics = {
    "total_requests": 0,
    "failed_requests": 0,
    "total_latency_ms": 0.0
}

@app.middleware("http")
async def monitor_requests(request: Request, call_next):
    start = time.time()
    metrics["total_requests"] += 1

    try:
        response = await call_next(request)
        if response.status_code >= 400:
            metrics["failed_requests"] += 1
        return response
    except Exception:
        metrics["failed_requests"] += 1
        raise
    finally:
        metrics["total_latency_ms"] += (
            time.time() - start
        ) * 1000

class PredictionInput(BaseModel):
    value: float

@app.get("/health")
def health():
    return {"status": "healthy"}

@app.post("/predict")
def predict(data: PredictionInput):
    prediction = int(model.predict([[data.value]])[0])
    probability = float(model.predict_proba([[data.value]])[0][1])

    return {
        "input": data.value,
        "prediction": prediction,
        "positive_probability": round(probability, 4)
    }

@app.get("/monitoring")
def monitoring():
    total = metrics["total_requests"]
    failed = metrics["failed_requests"]

    return {
        **metrics,
        "average_latency_ms": round(
            metrics["total_latency_ms"] / total, 2
        ) if total else 0,
        "failure_rate": round(failed / total, 4) if total else 0
    }
'''

(project_dir / "app.py").write_text(app_code.strip() + "\n")

print("Created app.py successfully!")

Created app.py successfully!


In [3]:
# W12D5: Automated API tests

test_code = '''
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

def test_health_endpoint():
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json()["status"] == "healthy"

def test_prediction_endpoint():
    response = client.post("/predict", json={"value": 4})
    assert response.status_code == 200
    assert response.json()["prediction"] == 1
    assert "positive_probability" in response.json()

def test_monitoring_endpoint():
    response = client.get("/monitoring")
    assert response.status_code == 200
    data = response.json()
    assert "total_requests" in data
    assert "average_latency_ms" in data
    assert "failure_rate" in data
'''

(project_dir / "tests" / "test_app.py").write_text(
    test_code.strip() + "\n"
)

print("Created tests/test_app.py successfully!")

Created tests/test_app.py successfully!


In [4]:
import subprocess

result = subprocess.run(
    ["python", "-m", "pytest", "-v"],
    cwd=str(project_dir),
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Tests failed. Share the error output.")

print("SUCCESS: All API tests passed!")

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/W12D5_Production_AI_System
plugins: anyio-4.15.1, platformdirs-4.12.2, langsmith-0.14.2, typeguard-4.6.0
collecting ... collected 3 items

tests/test_app.py::test_health_endpoint PASSED                           [ 33%]
tests/test_app.py::test_prediction_endpoint PASSED                       [ 66%]
tests/test_app.py::test_monitoring_endpoint PASSED                       [100%]

============================== 3 passed in 1.72s ===============================


SUCCESS: All API tests passed!


In [5]:
# W12D5: Dockerfile and dependencies

(project_dir / "requirements.txt").write_text(
    """fastapi
uvicorn
scikit-learn
pydantic
"""
)

(project_dir / "requirements-dev.txt").write_text(
    """pytest
httpx
ruff
"""
)

dockerfile = """
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

(project_dir / "Dockerfile").write_text(dockerfile.strip() + "\n")

gitignore = """
__pycache__/
*.py[cod]
.venv/
.env
.pytest_cache/
.ruff_cache/
"""

(project_dir / ".gitignore").write_text(gitignore.strip() + "\n")

print("Created requirements.txt")
print("Created requirements-dev.txt")
print("Created Dockerfile")
print("Created .gitignore")

Created requirements.txt
Created requirements-dev.txt
Created Dockerfile
Created .gitignore


In [7]:
# W12D5: Production monitoring and retraining strategy

monitoring_doc = """
# W12D5 — Production Monitoring Strategy

## 1. Metrics to Track
- API request volume and traffic patterns.
- API error rate and HTTP 4xx/5xx responses.
- Average and p95 response latency.
- CPU, memory, and container health.
- Input data quality and missing values.
- Prediction distribution and model performance.

## 2. Alerts
- Health endpoint failures: alert immediately.
- Server error rate above 5% over a monitoring window.
- p95 latency above the agreed service threshold.
- Sudden input feature or prediction distribution changes.
- Model performance below the required validation threshold.

## 3. Retraining Triggers
- Confirmed data or concept drift.
- Sustained decline in accuracy or other task metrics.
- Significant changes in incoming data.
- A scheduled review, if supported by fresh validated data.

Retraining must include data validation, model evaluation,
and approval before deployment. Keep the previous model
available for rollback.

## 4. CI/CD Pipeline
1. Install dependencies.
2. Run Ruff lint checks.
3. Run Pytest API tests.
4. Build the Docker image.
5. Publish the image to a registry when credentials
   and branch conditions permit.

## 5. Production Limitations
The current API stores metrics in memory, so they reset
when the process restarts. Production deployments should
use persistent monitoring, centralized logs, and an
external metrics backend. Drift thresholds must be
validated against representative production data.
"""

(project_dir / "monitoring.md").write_text(
    monitoring_doc.strip() + "\n"
)

print("Created monitoring.md successfully!")

Created monitoring.md successfully!


In [8]:
# W12D5: GitHub Actions CI/CD workflow

workflow = """
name: W12D5 Production AI Pipeline

on:
  push:
    branches: ["**"]
  pull_request:
    branches: ["**"]

jobs:
  lint-test-build:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt -r requirements-dev.txt

      - name: Lint code
        run: ruff check app.py tests

      - name: Run tests
        run: pytest -v

      - name: Build Docker image
        run: docker build -t w12d5-ml-api:${{ github.sha }} .

      - name: Log in to Docker Hub
        if: github.event_name == 'push' && github.ref == 'refs/heads/main'
        uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKERHUB_USERNAME }}
          password: ${{ secrets.DOCKERHUB_TOKEN }}

      - name: Push Docker image
        if: github.event_name == 'push' && github.ref == 'refs/heads/main'
        run: |
          docker tag w12d5-ml-api:${{ github.sha }} ${{ secrets.DOCKERHUB_USERNAME }}/w12d5-ml-api:${{ github.sha }}
          docker push ${{ secrets.DOCKERHUB_USERNAME }}/w12d5-ml-api:${{ github.sha }}
"""

workflow_dir = project_dir / ".github" / "workflows"
workflow_dir.mkdir(parents=True, exist_ok=True)
(workflow_dir / "ci.yml").write_text(workflow.strip() + "\n")

print("Created .github/workflows/ci.yml successfully!")

Created .github/workflows/ci.yml successfully!


In [9]:
import subprocess

# Run Ruff
lint_result = subprocess.run(
    ["python", "-m", "ruff", "check", "--fix", "app.py", "tests"],
    cwd=str(project_dir),
    capture_output=True,
    text=True
)

print("=== LINT ===")
print(lint_result.stdout)
print(lint_result.stderr)

if lint_result.returncode != 0:
    raise RuntimeError("Lint checks failed. Share the output.")

# Run tests
test_result = subprocess.run(
    ["python", "-m", "pytest", "-v"],
    cwd=str(project_dir),
    capture_output=True,
    text=True
)

test_output = test_result.stdout + "\n" + test_result.stderr
print("\n=== TESTS ===")
print(test_output)

outputs_dir = project_dir / "outputs"
(outputs_dir / "lint_results.txt").write_text(
    "All lint checks passed.\n"
)
(outputs_dir / "test_results.txt").write_text(test_output)

if test_result.returncode != 0:
    raise RuntimeError("Tests failed. Share the output.")

print("\nSUCCESS: Lint checks and tests passed!")

=== LINT ===
Found 2 errors (2 fixed, 0 remaining).



=== TESTS ===
============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/W12D5_Production_AI_System
plugins: anyio-4.15.1, platformdirs-4.12.2, langsmith-0.14.2, typeguard-4.6.0
collecting ... collected 3 items

tests/test_app.py::test_health_endpoint PASSED                           [ 33%]
tests/test_app.py::test_prediction_endpoint PASSED                       [ 66%]
tests/test_app.py::test_monitoring_endpoint PASSED                       [100%]

============================== 3 passed in 1.11s ===============================



SUCCESS: Lint checks and tests passed!


In [10]:
# W12D5: Generate API output files

import sys
import json
import importlib.util
from fastapi.testclient import TestClient

# Load this project's app.py directly
spec = importlib.util.spec_from_file_location(
    "w12d5_app", project_dir / "app.py"
)
w12d5_app = importlib.util.module_from_spec(spec)
spec.loader.exec_module(w12d5_app)

client = TestClient(w12d5_app.app)

# Call the API endpoints
health = client.get("/health")
prediction = client.post("/predict", json={"value": 4})
monitoring = client.get("/monitoring")

# Confirm all endpoints worked
assert health.status_code == 200
assert prediction.status_code == 200
assert monitoring.status_code == 200

# Save actual responses
outputs_dir = project_dir / "outputs"
outputs_dir.mkdir(exist_ok=True)

responses = {
    "health_output.json": health.json(),
    "prediction_output.json": prediction.json(),
    "monitoring_metrics.json": monitoring.json()
}

for filename, data in responses.items():
    (outputs_dir / filename).write_text(
        json.dumps(data, indent=2)
    )
    print(f"\n{filename}")
    print(json.dumps(data, indent=2))

print("\nSUCCESS: All three API outputs saved.")


health_output.json
{
  "status": "healthy"
}

prediction_output.json
{
  "input": 4.0,
  "prediction": 1,
  "positive_probability": 0.843
}

monitoring_metrics.json
{
  "total_requests": 3,
  "failed_requests": 0,
  "total_latency_ms": 3.519773483276367,
  "average_latency_ms": 1.17,
  "failure_rate": 0.0
}

SUCCESS: All three API outputs saved.


In [11]:
# W12D5: Verify submission files

required_files = [
    "app.py",
    "Dockerfile",
    "requirements.txt",
    "requirements-dev.txt",
    "monitoring.md",
    ".gitignore",
    "tests/test_app.py",
    ".github/workflows/ci.yml",
    "outputs/lint_results.txt",
    "outputs/test_results.txt",
    "outputs/health_output.json",
    "outputs/prediction_output.json",
    "outputs/monitoring_metrics.json",
]

missing = []

for filename in required_files:
    if (project_dir / filename).is_file():
        print("✅", filename)
    else:
        print("❌ MISSING:", filename)
        missing.append(filename)

if missing:
    print("\nMissing files:", missing)
else:
    print("\nSUCCESS: All W12D5 files are ready!")

✅ app.py
✅ Dockerfile
✅ requirements.txt
✅ requirements-dev.txt
✅ monitoring.md
✅ .gitignore
✅ tests/test_app.py
✅ .github/workflows/ci.yml
✅ outputs/lint_results.txt
✅ outputs/test_results.txt
✅ outputs/health_output.json
✅ outputs/prediction_output.json
✅ outputs/monitoring_metrics.json

SUCCESS: All W12D5 files are ready!


In [12]:
# W12D5: Download project files individually

from google.colab import files
import time

files_to_download = [
    "app.py",
    "Dockerfile",
    "requirements.txt",
    "requirements-dev.txt",
    "README.md",
    "monitoring.md",
    ".gitignore",
    "tests/test_app.py",
    ".github/workflows/ci.yml",
    "outputs/lint_results.txt",
    "outputs/test_results.txt",
    "outputs/health_output.json",
    "outputs/prediction_output.json",
    "outputs/monitoring_metrics.json",
]

for filename in files_to_download:
    path = project_dir / filename

    if path.is_file():
        print("Downloading:", filename)
        files.download(str(path))
        time.sleep(2)
    else:
        print("Missing:", filename)

print("Download requests completed!")

Downloading: app.py


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: Dockerfile


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Downloading: outputs/monitoring_metrics.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download requests completed!
